In [1]:
import pandas as pd
import numpy as np
import cplex

from sys import path
path.append("..") 
import matplotlib.pyplot as plt

from CPNorm import CorrOpti
from CPMini import CorrMini

In [2]:
#NORMALIZE SC ID-seq DATASET

df = pd.read_csv("/Users/t.stohn/Desktop/Projects/scIDseq_Niels/scIDseq-CNR/data/processed/scIDseq-venEijl-raw-counts.tsv", sep = "\t")
#df = pd.read_csv("/Users/t.stohn/Desktop/Projects_Paper/MRA/datasets/ID-seq/EGFRInh/RAW/REVII44_ag1478_countstbl.csv", sep = ",")

tmmBetaFactors = pd.read_csv("/Users/t.stohn/Desktop/Projects/scNormalization/CorrelationPreservationNorm/test/scIDseq_data/Evert_Klaas/tmmBetaFactors.tsv", sep = "\t")
print(df)

df = df[["sample_id","ab_name","ab_count_raw"]]
df = df.pivot(index="sample_id", columns="ab_name", values="ab_count_raw")

normalization = CorrMini(df)
#eta=1 means ONLY COVARIANCE minimization
normalization.print_terms()
normalization.solve(0.0005, 10, 0.01)
#print(normalization.get_logbeta_values())
normData = (normalization.get_normalized_data())

normData.to_csv( "/Users/t.stohn/Desktop/Projects/scNormalization/CorrelationPreservationNorm/test/scIDseq_data/Evert_Klaas/scIDseq_normalized.tsv", sep = "\t")

          sample_id plate_number    treatment ab_name ab_type  ab_count_raw
0      plate_10_171     plate_10  ip70S6K_EGF   ITGB1   total         21905
1      plate_12_171     plate_12       No_EGF   ITGB1   total         21300
2      plate_13_171     plate_13       No_EGF   ITGB1   total         21645
3       plate_2_171      plate_2  ip70S6K_EGF   ITGB1   total         33671
4       plate_3_171      plate_3     iRSK_EGF   ITGB1   total         19818
...             ...          ...          ...     ...     ...           ...
47605   plate_3_193      plate_3          EGF    JAK1   total          2181
47606   plate_4_193      plate_4       No_EGF    JAK1   total          3784
47607   plate_6_193      plate_6  ip70S6K_EGF    JAK1   total          2050
47608   plate_7_193      plate_7     iRSK_EGF    JAK1   total          2028
47609   plate_8_193      plate_8       No_EGF    JAK1   total          1704

[47610 rows x 6 columns]
ab_name       AKT123_P    AKT1_P      AKT2    BMP2_4    BMPRII

KeyboardInterrupt: 

In [ ]:
import seaborn as sns

betaFactors = normalization.get_beta_values()

print(betaFactors)
print(normData)

correlations = {}
for column in normData.columns:
    correlations[column] = np.corrcoef(normData[column], betaFactors)[0, 1]

# Convert the result to a DataFrame for a clearer view
correlations_df = pd.DataFrame(list(correlations.items()), columns=['Column', 'Correlation'])

sns.kdeplot(data=correlations_df, x='Correlation', fill=True)
plt.title('Density Plot of Correlations')
plt.xlabel('Correlation')
plt.ylabel('Density')
plt.show()